# Lab 3 · Graph Neural Networks: learning on an irregular sensor network — **30 minutes**

**LETMA 2026 Summer School · Lab 3 of 3** — Lab 1: FNO for Burgers · Lab 2: FFNO for infrasound.

**One scientific question.** *A seismic network is a handful of stations scattered over a map —
not a grid, not always the same stations, in no meaningful order. How do we make a network
respect that geometry: learn from it, remain correct when stations drop out, and avoid the
failure mode that plagues deep graph models — oversmoothing?*

| Time | Section | What you will see |
|---|---|---|
| 00–02 | §0 Setup | device, PyG import check |
| 02–08 | §1 How the graph is constructed | input distributions + **animation: k-NN edge selection, node by node** |
| 08–15 | §2 Message passing, defined precisely | **animation: information spreading hop by hop** + **Exercise G1 (6 min)** |
| 15–21 | §3 Oversmoothing: the deep-GNN trap | energy curves + the depth experiment + **animated energy collapse** |
| 21–27 | §4 Train the locator + **Exercise G2 (5 min)** | **animation: epicentres converging while training** |
| 27–30 | §5 Evaluate in kilometres · §6 takeaways | error maps, graph-free baseline, transfer checklist |

**How to finish in 30 minutes — read this before starting.** The critical path is the table
above; the ⏱ checkpoints at each section header tell you where you should be. Three safety
mechanisms keep everyone on track, whatever your ML background:

1. **Exercises cannot block you.** Each one has an automatic checker; if your attempt does not
   pass, the reference implementation takes over and the lab continues — a failed exercise
   costs only the minutes you spent on it, never the lab.
2. **The optional parts are short and skippable**: the depth experiment and the animations are
   quick to watch, and everything is already executed in the solution notebook.
3. **🧭 boxes are the novice path.** If you are new to ML: read the 🧭 boxes, watch the
   animations, do the exercise, and keep moving. The 📐 boxes are reference material to
   revisit afterwards — nobody is expected to absorb every formula live.

All the training on the critical path together takes seconds on the cluster GPU and under
~2 min on a laptop CPU: the clock is spent on reading and watching, not on waiting.

**How to use this notebook.** 🧭 boxes assume no ML background; 📐 boxes state every assumption
(H1, H2, …) precisely. Labs 1–2 (loss families, weight decay, landscapes) carry over unchanged;
this lab adds the *geometry* dimension and its specific pathology. Runs on Hedy or a laptop CPU;
animations are inline, press ▶ inside each widget.

## §0 · Setup — 00–02 min · ⏱ running by minute 2

---
**Authors:** E. Noëlé¹²³⁴, J.-P. Gatti⁵, L. Lehmann⁵, C. Millet⁵, D. Clouteau³

**Affiliations:**
¹ AMIAD — ² CEA — ³ LMPS (Laboratoire de Mécanique Paris-Saclay, CNRS/ENS
Paris-Saclay/CentraleSupélec/Université Paris-Saclay) — ⁴ Centre Borelli (ENS Paris-Saclay) —
⁵ CEA/DAM

**LETMA 2026 Summer School** · Machine Learning on Geophysical Data


In [ ]:
# Portable setup: runs unchanged on the Hedy cluster or on a laptop. Run this cell first.
import os, sys
from pathlib import Path
project_root = next((p for p in [Path(os.environ.get('HEDY_PROJECT_ROOT', Path.cwd())),
                                  Path.cwd(), *Path.cwd().parents]
                     if (p / 'fno_tuto').is_dir() and (p / 'gnn_tuto').is_dir()), None)
if project_root is None:
    raise RuntimeError('Open this notebook inside the tutorial project directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# LAB_PLOT_MODE=tk only for interactive SSH/X11 on Hedy (set before launching the kernel).
from IPython import get_ipython
PLOT_MODE = os.environ.get('LAB_PLOT_MODE', 'inline').strip().lower()
get_ipython().run_line_magic('matplotlib', PLOT_MODE)

import math, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

from fno_tuto.plotting import setup_style
setup_style()
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 12,
                     'axes.labelsize': 11, 'legend.fontsize': 10})
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

# Optional Hedy session profiling: active only on the cluster, silently skipped elsewhere.
HEDY_SESSION = None
try:
    from hedy_euporie.profiler import begin_session
    HEDY_SESSION = begin_session('lab_gnn_student', backend='torch')
except Exception:
    print('Hedy session profiling not available here (local/laptop run): continuing without it.')

def show_anim(fig, func, frames, interval=200, blit=False):
    """Inline JavaScript animation: works in Jupyter, JupyterLab and Euporie."""
    ani = animation.FuncAnimation(fig, func, frames=frames, interval=interval, blit=blit)
    plt.close(fig)          # suppress the empty static frame
    display(HTML(ani.to_jshtml()))

print(f'Python {sys.version.split()[0]} | torch {torch.__version__} | device {DEVICE}')

In [ ]:
try:
    import torch_geometric
    from torch_geometric.data import Batch, Data
    from torch_geometric.nn import MessagePassing, global_mean_pool
except ImportError as err:
    raise SystemExit('This lab needs torch_geometric (pip install torch_geometric)') from err

from gnn_tuto.seismic import make_network, make_events, azimuthal_gap
from gnn_tuto.graphs import (event_to_graph, events_to_graphs, events_to_vectors,
                             pred_to_hypo, hypo_errors, error_summary)
from gnn_tuto.models import LocGNN, MessageLayer, StationMLP, mlp
from gnn_tuto.td import check_message_layer, check_loc_gnn
print('imports OK | torch_geometric', torch_geometric.__version__)

## §1 · From a sensor network to a graph — 02–08 min · ⏱ checkpoint: start §2 at minute 8

**The task.** An earthquake occurs somewhere in a 100×100 km region; a network of 30 surface
stations picks the P arrival (and, 70% of the time, the S arrival) with noise; each event is seen
by a *random subset* of stations (0–30% are down). From those heterogeneous, unordered
observations we want the **hypocentre**: epicentre $(x, y)$ and depth $z$. No grid exists here —
only points and distances. That is the native habitat of a graph.

**A graph, defined once.** $G = (V, E)$: **nodes** $V$ (here: the stations that recorded the
event) with features $h_i \in \mathbb R^d$, and **edges** $E$ (here: $j \to i$ when $j$ is among
the 6 nearest neighbours of $i$), plus a feature matrix and an adjacency structure — nothing more.

**How OUR graph is built, cell by cell (this is the transferable recipe):**

- **Nodes**: the stations that recorded the event — the node *set itself changes per event*.
- **Node features** (5 numbers, all relative — see H6):
  1–2. position relative to the **reference station** (the first P arrival), in units of 20 km;
  3. P arrival time relative to the first P arrival, in units of 10 s (the origin time is unknown!);
  4. S arrival time relative to the first P arrival (0 if not picked);
  5. S-pick indicator (0/1).
- **Edges**: $k$-nearest-neighbour ($k=6$) in position. Why kNN and not "all pairs"? $O(nk)$
  instead of $O(n^2)$, and no spurious long-range shortcuts — the model must *route* distant
  information through intermediate stations, like waves do.
- **Target**: hypocentre relative to the reference station (and depth relative to 15 km), in
  units of 20 km — a single scale on all three axes, so the MSE loss weighs them equally.

**Every hypothesis, explicitly:**

- **H1 (Earth model).** Velocities $v_P(z) = 5.5 + 0.03\,z$ km/s (and a fixed $v_P/v_S$ ratio):
  a known, smooth 1-D model — no 3-D structure, no anisotropy.
- **H2 (pick noise).** Gaussian: $\sigma_P = 0.05$ s, $\sigma_S = 0.10$ s; S picks present with
  probability 0.7 (masked, not garbage).
- **H3 (availability).** Each event loses 0–30% of stations (at least 6 remain): the node count
  and identity vary per event.
- **H4 (sampling).** Epicentres uniform in the region, depths uniform on [1, 30] km, i.i.d.;
  splits by event index; the test events are touched once, in §5.
- **H5 (connectivity).** kNN with $k=6$ connects the observations "well enough" — a hypothesis we
  can check: with 30 nodes and $k=6$ the graph is essentially always connected.
- **H6 (invariance).** The physics is invariant to a global translation of space and to the
  choice of time origin; the features above bake that in (everything is *relative* to the
  reference station). The learning problem is then well-posed without absolute coordinates.
- **H7 (metric).** Training loss: MSE on the normalized target (each axis / 20 km); reported
  errors: **kilometres** (§5).

> 🧭 **First time with graphs?** A graph neural network is just: (1) every node starts with its
> own measurements; (2) in each *layer*, every node sends a **message** to its graph neighbours
> and receives the average of theirs; (3) each node updates itself from its own state and the
> average it received. Repeat 2–4 times, then aggregate node states into one prediction per event.
>
> 📐 **Invariance, stated properly.** Let $\sigma$ be any relabeling of stations. Our target must be
> *invariant* (the hypocentre does not depend on station numbering) and the node states
> *equivariant* (relabel inputs → outputs relabel). Mean-pooling of equivariant states gives an
> invariant readout: $\mathrm{mean}(\sigma h) = \mathrm{mean}(h)$. All of §2 is built to satisfy
> exactly this — it is what lets the model treat 26 stations one day and 30 the next.

In [ ]:
# ---- build the network, one event, one graph: look at the actual numbers ----
NETWORK = make_network(30, seed=2)
demo_events = make_events(NETWORK, 3, seed=100)
demo_graph = event_to_graph(demo_events[0], k=6)
print('graph:', demo_graph)
print()
print('node features (first 6 stations):  dx/20km  dy/20km  dP/10s  dS/10s  S-picked')
for row in demo_graph.x[:6]:
    print('  ', '  '.join(f'{v:+.3f}' for v in row.tolist()))
indeg = torch.bincount(demo_graph.edge_index[1], minlength=demo_graph.num_nodes)
outdeg = torch.bincount(demo_graph.edge_index[0], minlength=demo_graph.num_nodes)
print(f'\nin-degree of every node = 6 (by construction) | out-degree varies: '
      f'min {outdeg.min().item()}, max {outdeg.max().item()}')

fig, ax = plt.subplots(figsize=(5.6, 5))
ax.scatter(NETWORK[:, 0], NETWORK[:, 1], s=18, c='0.6', label='network (30 stations)')
seen = demo_graph.station_ids.tolist()
ax.scatter(NETWORK[seen, 0], NETWORK[seen, 1], s=45, c='C0', label='stations recording this event')
ax.scatter(*demo_events[0].hypo[:2], marker='*', s=180, c='C3', label='true hypocentre (plan)')
for s, t in demo_graph.edge_index.t().tolist():
    ax.plot([NETWORK[seen[s], 0].item(), NETWORK[seen[t], 0].item()],
            [NETWORK[seen[s], 1].item(), NETWORK[seen[t], 1].item()],
            color='C0', alpha=0.3, lw=0.7)
ax.set_xlabel('x (km)'); ax.set_ylabel('y (km)'); ax.legend(fontsize=8, loc='upper right')
ax.set_title('one event = one graph on the recorded stations'); ax.grid(alpha=0.3)
plt.show()

In [ ]:
# ---- the statistical distributions of the INPUTS (know your data!) ----
stats_events = make_events(NETWORK, 500, seed=999)
stats_graphs = events_to_graphs(stats_events)
feat_all = torch.cat([g.x for g in stats_graphs])            # every node feature of 500 events
fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.2))
ax[0].hist(feat_all[:, 2].numpy(), bins=40, color='C0', alpha=0.85)
ax[0].set_xlabel('relative P arrival time (units of 10 s)')
ax[0].set_ylabel('count')
ax[0].set_title('INPUT: node feature dP (the pick the GNN will read)')
ax[1].hist([len(e.station_ids) for e in stats_events], bins=np.arange(5.5, 32, 1),
           color='C1', alpha=0.85)
ax[1].set_xlabel('stations recording the event')
ax[1].set_title('INPUT: node count per event (H3) - the graph size changes')
ax[2].hist([float(e.hypo[2]) for e in stats_events], bins=30, color='C2', alpha=0.85)
ax[2].set_xlabel('earthquake depth (km)')
ax[2].set_title('TARGET: depth distribution (H4)')
plt.tight_layout(); plt.show()
print(f'node features: {feat_all.shape[0]} stations-events pairs | '
      f'P picks range over {feat_all[:, 2].max() - feat_all[:, 2].min():.1f} x 10 s')

In [ ]:
# ---- animation 1: how k-NEAREST-NEIGHBOURS builds the graph, node by node ----
from matplotlib.collections import LineCollection
n_seen = len(seen)
pos_np = NETWORK[seen].numpy()[:, :2]                       # absolute station positions (x, y in km)
K_NB = 6
dist_mat = np.linalg.norm(pos_np[:, None, :] - pos_np[None, :, :], axis=-1)
order = np.argsort(dist_mat, axis=1)                         # neighbours of each node, nearest first

fig, ax = plt.subplots(figsize=(6.8, 6.0))
ax.set_xlim(-4, 104); ax.set_ylim(-4, 104); ax.set_xlabel('x (km)'); ax.set_ylabel('y (km)')
ax.grid(alpha=0.3)
ax.scatter(*NETWORK[:, :2].T, s=16, c='0.85', zorder=1)      # stations not recording this event
feat = demo_graph.x[:, 2]                                    # P-time feature for the final frame
sc_seen = ax.scatter(pos_np[:, 0], pos_np[:, 1], s=55, c=np.zeros(n_seen),
                     cmap='coolwarm', vmin=-0.1, vmax=float(feat.max()), zorder=3)
(active_pt,) = ax.plot([], [], 'o', color='C3', ms=13, zorder=4)
lc_cand = LineCollection([], color='0.6', lw=0.6, alpha=0.4, zorder=2)
lc_select = LineCollection([], color='C3', lw=2.0, zorder=2)
lc_commit = LineCollection([], color='C0', lw=0.9, alpha=0.7, zorder=2)
for lc in (lc_cand, lc_select, lc_commit):
    ax.add_collection(lc)
hy, = ax.plot([], [], marker='*', ms=18, c='k', ls='none', zorder=5)
title = ax.set_title('')

committed = []
def update(f):
    if f < n_seen:
        i = f
        others = [j for j in range(n_seen) if j != i]
        lc_cand.set_segments([[pos_np[i], pos_np[j]] for j in others])
        sel = [order[i, 1 + k] for k in range(min(K_NB, n_seen - 1))]
        lc_select.set_segments([[pos_np[i], pos_np[j]] for j in sel])
        committed.extend([[pos_np[i], pos_np[j]] for j in sel])
        lc_commit.set_segments(list(committed))
        active_pt.set_data([pos_np[i, 0]], [pos_np[i, 1]])
        title.set_text(f'node {i + 1}/{n_seen}: grey = all candidates, red = its {K_NB} '
                       'nearest -> edges committed (blue)')
    else:
        lc_cand.set_segments([]); lc_select.set_segments([]); active_pt.set_data([], [])
        sc_seen.set_array(feat.numpy())
        hy.set_data([demo_events[0].hypo[0].item()], [demo_events[0].hypo[1].item()])
        title.set_text('graph complete: colors = relative P arrival time; star = hypocentre to locate')
show_anim(fig, update, frames=n_seen + 1, interval=420)

> 🧭 **What to notice.** The construction is *three decisions*: what is a node (a recording
> station), what is an edge (6 nearest neighbours), what are the features (relative positions and
> times). Everything else in this lab is machinery. When you apply a GNN to your own data, these
> three decisions are where the science lives — state them as hypotheses (here H3, H5, H6).
>
> 📐 **Why relative features (H6) are not a cosmetic choice.** With absolute positions the
> learning problem would have to discover translation invariance from data; with relative features
> the hypothesis class *only contains* translation-invariant functions. Smaller hypothesis class,
> same target: strictly better sample complexity, by construction.

## §2 · Message passing, defined precisely — 08–15 min · ⏱ checkpoint: start §3 at minute 15

One layer of our GNN, in full (this is the exact equation the exercise implements):

$$m_{ij} = M_\theta\big(h_i,\, h_j,\, p_j - p_i,\, \lVert p_j - p_i \rVert\big) \quad
\text{(message from } j \text{ to } i), \qquad
a_i = \frac{1}{|\mathcal N(i)|} \sum_{j \in \mathcal N(i)} m_{ij}, \qquad
h_i \leftarrow h_i + U_\theta\big(h_i, a_i\big).$$

Three properties worth memorizing:

1. **Permutation invariance of $a_i$**: sum/mean/max are the only aggregations that treat
   neighbours unordered — a network with $\Sigma$ (sum) would change when stations are relabeled.
2. **Locality per layer, globality by depth**: after $\ell$ layers, node $i$ has seen
   $\ell$-hop neighbourhoods. With $k=6$ neighbours our 30-node graphs need $\approx$ 3–4 hops to
   mix — that sets the depth (§3 makes this quantitative).
3. **Relative geometry in the message** ($p_j - p_i$, not $p_j$): equivariance to translation again.

**Exercise G1 · 6 minutes.** Implement `message` and `forward` of the layer (the `MessagePassing`
base class handles the scatter-mean for you). The checker compares against a reference;
if it fails, the reference takes over and the lab continues. ⏱ hard stop at minute 15: not
passing yet? Move on — the fallback keeps the lab on track.

In [ ]:
import torch.nn as nn
from torch_geometric.nn import MessagePassing, global_mean_pool
from gnn_tuto.models import mlp

class MyMessageLayer(MessagePassing):
    """Messages j -> i, mean aggregation, residual node update."""
    def __init__(self, hidden, pos_dim=2):
        super().__init__(aggr='mean')
        self.msg = mlp(2*hidden + pos_dim + 1, hidden, hidden)
        self.upd = mlp(2*hidden, hidden, hidden)

    def message(self, h_i, h_j, pos_i, pos_j):
        # TODO G1: rel = pos_j - pos_i; concatenate h_i, h_j, rel, ||rel||.
        # The norm must preserve the final singleton dimension.
        raise NotImplementedError('Complete message()')

    def forward(self, h, pos, edge_index):
        # TODO G1: self.propagate(...) aggregates messages into targets.
        # Apply the residual update h + upd(concat(h, aggregated)).
        raise NotImplementedError('Complete forward()')

In [ ]:
passed_g1 = check_message_layer(MyMessageLayer)
LayerForLab = MyMessageLayer if passed_g1 else MessageLayer
print('G1:', 'your message layer is active' if passed_g1 else 'reference fallback active')

In [ ]:
# ---- animation 2: information spreading hop by hop (untrained weights, one event) ----
torch.manual_seed(3)
ENC = mlp(5, 64, 64)                                              # fixed random encoder
layers_mp = torch.nn.ModuleList([MessageLayer(64) for _ in range(4)])
with torch.no_grad():
    demo_h = torch.nn.functional.gelu(ENC(demo_graph.x))
    states = [demo_h]
    h = demo_h
    for layer in layers_mp:
        h = layer(h, demo_graph.pos, demo_graph.edge_index)
        states.append(h)
proj = torch.randn(64); proj /= proj.norm()                       # fixed random projection to 1D
vals = [(st @ proj).numpy() for st in states]
vmax = max(abs(v).max() for v in vals)
LBL = ['encoder (own features only)'] + [f'after layer {i}' for i in range(1, 5)]

fig, ax = plt.subplots(figsize=(6.4, 5.4))
ax.set_xlim(-4, 104); ax.set_ylim(-4, 104); ax.grid(alpha=0.3)
ax.set_xlabel('x (km)'); ax.set_ylabel('y (km)')
sc = ax.scatter(NETWORK[seen, 0], NETWORK[seen, 1], c=vals[0], cmap='coolwarm',
                s=70, vmin=-vmax, vmax=vmax)
fig.colorbar(sc, ax=ax, shrink=0.8, label='projected node state (untrained)')
title = ax.set_title(LBL[0])

def update(f):
    sc.set_array(vals[f])
    title.set_text(LBL[f])
    return sc, title

show_anim(fig, update, frames=len(vals), interval=650)
print('Watch the palette MIX: by layer 3-4 the whole network shares one picture of the event.')
print('The next section shows what happens when you keep going: too many layers erase the detail.')

## §3 · Oversmoothing: the deep-GNN trap — 15–21 min · ⏱ checkpoint: start §4 at minute 21

**Definition.** Oversmoothing is the convergence of node representations to a common vector as
the number of message-passing layers grows: $h_i^{(\ell)} \to \bar h$ for all $i$ — the node
features lose the very information that distinguishes nodes, and the readout sees the same
average regardless of the input.

**The mechanism, in one line of spectral graph theory.** Mean-aggregation applies the smoothing
operator $P = D^{-1}A$ (random-walk normalized adjacency): for any signal $s$ on the nodes,

$$P^\ell s = \sum_k (1 - \lambda_k)^\ell\, \langle s, \phi_k\rangle\, \phi_k,$$

where $\lambda_k$ are the eigenvalues of the graph Laplacian: high-frequency components
($\lambda_k$ away from 0) decay geometrically, $\bar s$ ($\lambda = 0$) survives. **Repeated
averaging is a low-pass filter on the graph** — depth beyond what information transport requires
only erases detail.

**Quantified with the Dirichlet energy** $E(h) = \sum_i \lVert h_i - \bar h \rVert_2^2$:
oversmoothing is $E \to 0$.

> 📐 **Known results** (Li, Han & Wu, AAAI 2018, and follow-ups): for GCN-type layers with ReLU
> and mean/sum aggregation on connected graphs, node features converge to the same vector and the
> Dirichlet energy decays *exponentially* in depth under mild conditions. Our layer softens this
> with a residual ($h \leftarrow h + U(\cdot)$) and GELU — it decays more slowly, but the
> mechanism is identical, and the depth experiment below shows the practical damage. Residuals
> delay oversmoothing; they do not repeal it.

**Remedies, in order of practicality**: (1) keep the depth at what the task needs
($\ell \approx$ graph diameter, here 3–4); (2) residual updates (already in our layer);
(3) read the *initial* features too (jumping knowledge: concatenate $h^{(0)}\dots h^{(L)}$);
(4) normalization per layer (GraphNorm-style). If your GNN degrades when you deepen it, you are
almost certainly looking at oversmoothing, not "not enough parameters".

In [ ]:
# ---- the mechanism isolated: pure repeated averaging vs our residual layer ----
n = demo_graph.num_nodes
pos = demo_graph.pos
A = torch.zeros(n, n)
A[demo_graph.edge_index[1], demo_graph.edge_index[0]] = 1.0        # target receives
P = A / A.sum(1, keepdim=True)                                     # mean-aggregation operator

g = torch.Generator().manual_seed(5)
signal = torch.randn(n, 8, generator=g)                            # arbitrary node signal
E_pure = []
s = signal.clone()
for _ in range(16):
    E_pure.append(((s - s.mean(0)).pow(2).sum()).item())
    s = P @ s                                                       # pure averaging (GCN mechanism)

E_res = []
torch.manual_seed(7)
h = torch.nn.functional.gelu(mlp(5, 64, 64)(demo_graph.x))
deep = torch.nn.ModuleList([MessageLayer(64) for _ in range(16)])
with torch.no_grad():
    for layer in deep:
        E_res.append(((h - h.mean(0)).pow(2).sum()).item())
        h = layer(h, pos, demo_graph.edge_index)

fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.2))
ax[0].semilogy(E_pure, 'o-', color='C3', label=r'pure mean-aggregation $P^\ell s$')
ax[0].semilogy(E_res, 's-', color='C0', label='our residual GELU layer')
ax[0].set_xlabel('layer (depth)'); ax[0].set_ylabel('Dirichlet energy $E(h)$')
ax[0].legend(fontsize=9); ax[0].grid(alpha=0.3)
ax[0].set_title('oversmoothing: energy -> 0 with depth (residuals only slow it down)')
ax[1].plot(E_pure, 'o-', color='C3'); ax[1].plot(E_res, 's-', color='C0')
ax[1].set_yscale('log'); ax[1].set_ylim(max(min(E_res)*0.3, 1e-3), max(E_pure)*3)
ax[1].set_xlim(0, 8); ax[1].set_xlabel('layer'); ax[1].set_ylabel('energy')
ax[1].grid(alpha=0.3); ax[1].set_title('zoom on the first layers: 3-4 hops mix the graph well')
plt.show()
print('After ~4 layers the mixing is essentially done: deeper layers average what is already '
      'an average. Depth beyond the diameter buys nothing and costs information.')

In [ ]:
# ---- animation 3: oversmoothing made quantitative: node states + the energy curve ----
with torch.no_grad():
    h = torch.nn.functional.gelu(ENC(demo_graph.x))
    states_deep, E_deep = [(h @ proj).numpy()], [((h - h.mean(0)).pow(2).sum()).item()]
    for layer in deep:
        h = layer(h, pos, demo_graph.edge_index)
        states_deep.append((h @ proj).numpy())
        E_deep.append(((h - h.mean(0)).pow(2).sum()).item())

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.8), gridspec_kw={'width_ratios': [1.05, 1]})
sc = ax[0].scatter(NETWORK[seen, 0], NETWORK[seen, 1], c=states_deep[0], cmap='coolwarm',
                   s=70, vmin=-vmax, vmax=vmax)
fig.colorbar(sc, ax=ax[0], shrink=0.8, label='projected node state (fixed color scale)')
ax[0].set_xlim(-4, 104); ax[0].set_ylim(-4, 104); ax[0].grid(alpha=0.3)
ax[0].set_xlabel('x (km)'); ax[0].set_ylabel('y (km)')
(lnE,) = ax[1].plot([], [], 'o-', color='C0', lw=1.6)
ax[1].set_yscale('log'); ax[1].set_xlim(0, len(E_deep) - 1)
ax[1].set_ylim(max(min(E_deep) * 0.2, 1e-2), max(E_deep) * 2.5)
ax[1].set_xlabel('depth (message-passing layers)')
ax[1].set_ylabel('Dirichlet energy E(h)  (log)')
ax[1].grid(alpha=0.3)
title = ax[0].set_title('depth 0 (encoder)')

def update(f):
    sc.set_array(states_deep[f])
    lnE.set_data(np.arange(f + 1), E_deep[:f + 1])
    title.set_text(f'depth {f}: colors homogenize AND the energy falls: '
                   f'E = {E_deep[f]:.1f}')
    return sc, lnE, title

show_anim(fig, update, frames=len(states_deep), interval=550)

> 🧭 **Watch the colors collapse.** At depth 4 you can still tell stations apart; by depth 10
> they are interchangeable — the network can no longer tell *which* station had the early
> arrival, so it cannot locate anything. Deep is not better on graphs.

### Does depth hurt the trained locator? A five-point experiment

In [ ]:
# ---- depth experiment: train 1, 2, 4, 8 layers and compare (short budgets, same data) ----
train_events = make_events(NETWORK, 200, seed=101)
val_events   = make_events(NETWORK, 40,  seed=202)
train_graphs = events_to_graphs(train_events)
val_graphs   = events_to_graphs(val_events)

def train_short(n_layers, epochs=40, seed=0):
    torch.manual_seed(seed)
    model = LocGNN(n_layers=n_layers)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    gen = torch.Generator().manual_seed(seed)
    for ep in range(epochs):
        perm = torch.randperm(len(train_graphs), generator=gen)
        for i in range(0, len(train_graphs), 32):
            b = Batch.from_data_list([train_graphs[j] for j in perm[i:i+32]])
            opt.zero_grad()
            loss = (model(b) - b.y).pow(2).mean()
            loss.backward(); opt.step()
    with torch.no_grad():
        errs = []
        for i in range(0, len(val_graphs), 64):
            b = Batch.from_data_list(val_graphs[i:i+64])
            errs.append((model(b) - b.y).pow(2).sum(1))
        return torch.cat(errs).sqrt().mean().item()

depths = [1, 2, 4, 8, 16]
val_err = {}
t0 = time.time()
for d_ in depths:
    val_err[d_] = train_short(d_)
    print(f'LocGNN with {d_} layer(s): validation error (normalized) = {val_err[d_]:.4f}')
print(f'depth experiment: {time.time()-t0:.0f} s')

fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(depths, [val_err[d] for d in depths], 'o-', color='C0')
ax.set_xlabel('number of message-passing layers'); ax.set_ylabel('validation error')
ax.grid(alpha=0.3)
ax.set_title('Depth: most of the gain by the graph diameter (~4); degradation appears at 16')
plt.show()

**Read this experiment honestly — it is the interesting outcome.** On this task, our *residual*
architecture delays oversmoothing: depth keeps helping slightly up to ~12 layers, and the
degradation only appears at 16. Compare with the **red curve of the energy plot**: pure
mean-aggregation (the GCN mechanism, without residual updates and learned mixing) collapses in a
handful of layers. So the correct mental model is: *oversmoothing is a property of repeated
aggregation, residuals and learned updates slow it down, and depth beyond what information
transport needs buys little and eventually costs accuracy and training time.* When your GNN gets
worse as you deepen it, measure the Dirichlet energy per layer first — it distinguishes
oversmoothing (energy $\to 0$) from mere optimization difficulty (energy stays high, loss
stalls).

## §4 · Train the locator — 21–27 min · ⏱ checkpoint: start §5 at minute 27

The full pipeline: **encode** each station's 5 features to a 64-dim state → **process** with 4
message-passing layers (states stay per-station, equivariant) → **pool** the node states of one
event into a single vector by a mean (invariant; `data.batch` tells PyG which nodes belong to
which event) → **decode** to the 3 normalized hypocentre coordinates. Loss: MSE — with the
single 20 km scale on all three axes (H7), the three error components are weighted equally by
construction; changing those scales is exactly a "hyperparameter of the loss" (Labs 1–2, §4).

**Where is the computation? In a latent space, as in Labs 1–2.** The encoder embeds each
station's 5 physical numbers into a 64-dimensional learned state; every layer, every message,
every aggregation acts on these latent states — the physics never reappears in the middle of the
network — and the mean-pooled latent state of the event is the *event code* from which the
decoder reads the hypocentre. Two properties make this representation special here:
(equivalence) permuting stations permutes the latent states, so the pooled code — and hence the
prediction — is invariant for free; (locality of information) each latent state summarizes one
station and its graph neighbourhood, and depth widens that neighbourhood — §2's map animation
was exactly a projection of this latent space, and the color homogenization you saw there is
§3's oversmoothing seen from inside it.

**Exercise G2 · 5 minutes.** Assemble the full model (encode → layers → pool → decode).
⏱ hard stop: not passing yet? The checker falls back to the reference LocGNN — move on.

In [ ]:
class MyLocGNN(nn.Module):
    def __init__(self, in_dim=5, hidden=64, n_layers=4, out_dim=3):
        super().__init__()
        self.encoder = mlp(in_dim, hidden, hidden)
        self.layers = nn.ModuleList([MyMessageLayer(hidden) for _ in range(n_layers)])
        self.decoder = mlp(hidden, hidden, out_dim)

    def forward(self, data):
        # TODO G2: encode station features, apply all graph layers,
        # aggregate to ONE vector PER GRAPH (data.batch), decode hypocentres.
        raise NotImplementedError('Complete MyLocGNN.forward')

In [ ]:
passed_g2 = check_loc_gnn(MyLocGNN)
GNNForLab = MyLocGNN if (passed_g1 and passed_g2) else LocGNN
print('G2:', 'your full GNN is active' if (passed_g1 and passed_g2)
      else 'reference fallback active (LocGNN)')

In [ ]:
# ---- training, with snapshots of test-event predictions for the animation ----
test_events  = make_events(NETWORK, 50, seed=303)
test_graphs  = events_to_graphs(test_events)
torch.manual_seed(0)
locator = GNNForLab(n_layers=4)
opt = torch.optim.Adam(locator.parameters(), lr=1e-3)
EPOCHS = 60
snaps = []
gen = torch.Generator().manual_seed(0)
hist = []
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    locator.train()
    perm = torch.randperm(len(train_graphs), generator=gen)
    for i in range(0, len(train_graphs), 32):
        b = Batch.from_data_list([train_graphs[j] for j in perm[i:i+32]])
        opt.zero_grad()
        loss = (locator(b) - b.y).pow(2).mean()
        loss.backward(); opt.step()
    locator.eval()
    with torch.no_grad():
        errs = []
        for i in range(0, len(val_graphs), 64):
            b = Batch.from_data_list(val_graphs[i:i+64])
            errs.append((locator(b) - b.y).pow(2).sum(1))
        hist.append(torch.cat(errs).sqrt().mean().item())
    if ep % 5 == 0 or ep == 1:
        with torch.no_grad():
            b = Batch.from_data_list(test_graphs[:12])
            snaps.append((ep, locator(b).detach().numpy(), b.ref.numpy()))
print(f'training done in {time.time()-t0:.0f} s | final validation error {hist[-1]:.4f} '
      f'(normalized; 1.0 ~ 20 km error)')
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(hist, color='C0'); ax.set_xlabel('epoch'); ax.set_ylabel('validation error')
ax.grid(alpha=0.3); ax.set_title('the locator converges in about a minute on a laptop CPU')
plt.show()

In [ ]:
# ---- animation 4: predicted hypocentres converging to the true ones, epoch by epoch ----
from gnn_tuto.graphs import pred_to_hypo
fig, ax = plt.subplots(1, 2, figsize=(11.5, 5), gridspec_kw={'width_ratios': [3, 1]})
ax[0].scatter(NETWORK[:, 0], NETWORK[:, 1], s=14, c='0.8', zorder=1)
ax[0].set_xlim(-4, 104); ax[0].set_ylim(-4, 104); ax[0].grid(alpha=0.3)
ax[0].set_xlabel('x (km)'); ax[0].set_ylabel('y (km)')
true_xy = torch.stack([e.hypo[:2] for e in test_events[:12]]).numpy()
ax[0].scatter(true_xy[:, 0], true_xy[:, 1], marker='*', s=160, c='C0', zorder=3,
              label='true epicentres')
(pred_pts,) = ax[0].plot([], [], 'x', ms=9, c='C3', zorder=4, label='GNN predictions')
ax[0].legend(fontsize=8, loc='upper right')
(loss_ln,) = ax[1].plot([s[0] for s in snaps], [hist[s[0] - 1] for s in snaps], color='0.4')
(dot,) = ax[1].plot([], [], 'o', c='C3')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('val error'); ax[1].grid(alpha=0.3)
title = ax[0].set_title('')

def update(f):
    ep, pred, ref = snaps[f]
    km = pred_to_hypo(torch.tensor(pred), torch.tensor(ref)).numpy()
    pred_pts.set_data(km[:, 0], km[:, 1])
    dot.set_data([ep], [hist[ep - 1]])
    title.set_text(f'epoch {ep}: predicted epicentres (x) vs truth (stars), 12 held-out events')

show_anim(fig, update, frames=len(snaps), interval=420)

## §5 · Evaluate in kilometres — 27–30 min · ⏱ wrap-up by minute 30

Normalized MSE is for the optimizer; geophysicists report **kilometres**. We convert the
normalized outputs back to physical hypocentres, and also train the *graph-free* baseline
(a plain MLP on a fixed 30-station vector with zeros for missing stations) to check that the
graph structure actually earns its keep (H3 is precisely what should hurt the MLP).

In [ ]:
# ---- final evaluation on the 50 untouched test events ----
locator.eval()
with torch.no_grad():
    preds = []
    for i in range(0, len(test_graphs), 64):
        b = Batch.from_data_list(test_graphs[i:i+64])
        preds.append(locator(b))
    pred_norm = torch.cat(preds)
refs = torch.cat([g.ref for g in test_graphs])
pred_km = pred_to_hypo(pred_norm.cpu(), refs.cpu())                      # (n, 3) in km
errs = (pred_km - torch.stack([e.hypo.float() for e in test_events])).abs()
epi = torch.norm(errs[:, :2], dim=1)                                     # horizontal error (km)
depth = errs[:, 2]
print(f'epicentral error: median {epi.median():.2f} km | mean {epi.mean():.2f} | 90% {epi.quantile(0.9):.2f} km')
print(f'depth error:       median {depth.median():.2f} km | mean {depth.mean():.2f} | 90% {depth.quantile(0.9):.2f} km')
print('per-axis summary:', error_summary(errs))

# ---- the graph-free baseline on the SAME events ----
vec_train, y_train = events_to_vectors(train_events, 30)
vec_test, y_test = events_to_vectors(test_events, 30)
torch.manual_seed(0)
mlp_base = StationMLP(30)
opt = torch.optim.Adam(mlp_base.parameters(), lr=1e-3)
for ep in range(60):
    perm = torch.randperm(len(vec_train))
    for i in range(0, len(vec_train), 32):
        idx = perm[i:i+32]
        opt.zero_grad()
        loss = (mlp_base(vec_train[idx]) - y_train[idx]).pow(2).mean()
        loss.backward(); opt.step()
with torch.no_grad():
    pred_mlp_km = pred_to_hypo(mlp_base(vec_test), refs)
errs_mlp = (pred_mlp_km - torch.stack([e.hypo.float() for e in test_events])).abs()
epi_mlp = torch.norm(errs_mlp[:, :2], dim=1)
print(f'\nbaseline MLP (fixed 30-station vector): epicentral median {epi_mlp.median():.2f} km '
      f'| depth median {errs_mlp[:, 2].median():.2f} km')
print('the graph handles station dropout (H3) gracefully; the fixed-vector MLP pays for every missing station')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(epi, bins=20, alpha=0.6, color='C0', label=f'GNN (median {epi.median():.1f} km)')
ax[0].hist(epi_mlp, bins=20, alpha=0.6, color='C3', label=f'MLP baseline (median {epi_mlp.median():.1f} km)')
ax[0].set_xlabel('epicentral error (km)'); ax[0].legend(fontsize=9); ax[0].grid(alpha=0.3)
gaps = [azimuthal_gap(e) for e in test_events]
ax[1].scatter(gaps, epi, s=14, c='C0')
ax[1].set_xlabel('azimuthal gap of the event (degrees)'); ax[1].set_ylabel('epicentral error (km)')
ax[1].grid(alpha=0.3); ax[1].set_title('geometry matters: events poorly surrounded by stations are harder')
plt.show()

**Reading the evaluation honestly.** Depth errors exceed horizontal ones — a surface-only array
constrains depth through moveout, the weakest part of the signal (a classic trade-off, inherited
from H1's 1-D model). Errors grow with the azimuthal gap: the network is doing *geometry-aware*
learning, and its failure modes are the geophysical ones. And remember H1–H4: this is a synthetic
world; on real picks you would re-check every hypothesis before trusting a number.

## §6 · Takeaways — and using GNNs on *your* problem

- **The graph is a modeling decision, not a file format.** Nodes / edges / features / invariances
  (our H3, H5, H6) are where your science enters; state them as hypotheses and check them.
- **Message passing = local aggregation, depth = range.** Set depth from the graph diameter, not
  from "deeper is better": §3 showed the energy collapse and the depth experiment agreed.
- **Oversmoothing is the graph-world's specific failure mode.** Diagnose it with the Dirichlet
  energy; fight it with depth discipline, residuals and jumping knowledge — and read your depth
  experiment honestly: here the residual update delayed the damage, the pure-averaging curve
  showed the mechanism immediately, and beyond the diameter the extra layers paid nothing.
- **Invariance by construction.** Relative features + permutation-invariant aggregation +
  pooling = a model that works for *any* number of sensors — the operational reason graphs beat
  fixed-size vectors in §5.

**Checklist: GNNs on your application.**
1. Define nodes (sensors? grid cells? particles?), edges (kNN? Delaunay? physical contact?) and
   features — write the invariances you believe in, then bake them into features and messages.
2. Start shallow: $L \approx$ the number of hops information must travel; add depth only with the
   energy diagnostic in hand.
3. Use mean pooling for intensive quantities (a hypocentre), sum pooling for extensive ones
   (total energy); the wrong pool is a silent physical error.
4. Train with Adam, MSE on a single well-chosen scale; evaluate in physical units.
5. Always run a graph-free baseline: the graph must earn its complexity.
6. If your problem lives on a regular grid after all, Lab 1–2 (FNO/FFNO) is the cheaper tool;
   use a graph when the geometry, connectivity or node set is genuinely irregular.

### Optional — save this interactive session measurement (Hedy only)
Same as Labs 1–2: a JSON timing report on the cluster, a one-line summary elsewhere.

## References

**Operator learning and FNO/FFNO:**
- Z. Li, N. Kovachki, K. Azizzadenesheli, B. Liu, K. Bhattacharya, A. Stuart, A. Anandkumar.
  *Fourier Neural Operator for Parametric Partial Differential Equations.* ICLR 2021.
- Z. Li, N. Kovachki, ... A. Anandkumar. *Neural Operator: Learning Maps Between Function
  Spaces.* Nature Machine Intelligence **6**, 666–676 (2024).
- A. Tran, A. Mathews, L. Xie, C.-K. Ong. *Factorized Fourier Neural Operators.* ICLR 2023.
- K. Azizzadenesheli, Z. Li, ... A. Anandkumar. *Neural Operators for Accelerating Scientific
  Simulations? Open Problems and Perspectives.* ICLR 2024 (position paper).

**Graph neural networks:**
- T. N. Kipf, M. Welling. *Semi-Supervised Classification with Graph Convolutional Networks.*
  ICLR 2017.
- J. Gilmer, S. S. Schoenholz, P. F. Riley, O. Vinyals, G. E. Dahl. *Neural Message Passing for
  Quantum Chemistry.* NeurIPS 2017.
- Q. Li, Z. Han, X.-M. Wu. *Deeper Insights into Graph Convolutional Networks for Semi-
  Supervised Learning.* AAAI 2018. (Oversmoothing and the Dirichlet-energy analysis.)
- R. L. Murphy, B. Srinivasan, V. Rao, B. Ribeiro. *Janossy Pooling: Learning Deep
  Permutation-Invariant Functions.* NeurIPS 2019. (Why permutation-invariant aggregation
  matters.)

**Loss landscapes and optimization:**
- H. Li, Z. Xu, G. Taylor, C. Studer, T. Goldstein. *Visualizing the Loss Landscape of Neural
  Nets.* NeurIPS 2018. (The 2-D random-slice methodology used in Lab 1 §4.)
- T. Garipov, P. Izmailov, D. Podoprikhin, D. Vetrov, A. Wilson. *Loss Surfaces, Mode
  Connectivity, and Fast Ensembling of DNNs.* NeurIPS 2018. (Why two trained networks are
  connected by a low-loss path — the multi-minimum geometry of Lab 1 §4.4.)

**Infrasound propagation and parabolic-equation methods:**
- A. Le Pichon, E. Blanc, A. Hauchecorne (eds.). *Infrasound Monitoring for Atmospheric
  Studies.* 2nd ed., Springer (2021).
- R. Waxler, J. Assink. *Propagation of infrasound through the atmosphere: theory and
  observations.* In: Infrasound Monitoring, ch. 5 (2019).
- D. P. Knupp, S. J. Franke. *A global reference atmosphere for infrasound propagation.*
  J. Acoust. Soc. Am. **140**, 2724 (2016).

**Software:**
- A. Paszke et al. *PyTorch: An Imperative Style, High-Performance Deep Learning Library.*
  NeurIPS 2019.
- M. Fey, J. E. Lenssen. *Fast Graph Representation Learning with PyTorch Geometric.*
  ICLR Workshop on Representation Learning on Graphs and Manifolds (2019).


In [ ]:
if HEDY_SESSION is not None:
    HEDY_SESSION.report()   # writes a JSON timing report on the Hedy cluster only
else:
    print('No Hedy session: nothing to report (laptop run).')